## Fire Occurrence Data Pipeline (NASA FIRMS)

# FireFusion – Fire Occurrence Dataset Preparation

## Sprint 1 Data Engineering Contribution
This notebook prepares a bushfire occurrence dataset using NASA FIRMS data for Australia.  
The output will support future bushfire prediction modelling by creating a cleaned dataset with:
- latitude
- longitude
- acquisition date
- confidence
- fire_occurred label

In [1]:
import pandas as pd
import numpy as np

## Load Source Data
The source dataset is the NASA FIRMS archive CSV file downloaded for Australia.
A subset is first loaded to inspect schema and validate columns.

In [2]:
file_path = "fire_archive_SV-C2_732345.csv"
df = pd.read_csv(file_path, nrows=10000)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'fire_archive_SV-C2_732345.csv'

## Inspect Dataset Structure
This step checks available columns and dataset shape before cleaning.

In [ ]:
print(df.columns)
df.shape

Index(['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'bright_t31', 'frp', 'daynight', 'type'],
      dtype='object')


(10000, 15)

## Select Relevant Columns
Only the fields required for bushfire occurrence modelling are retained:
- latitude
- longitude
- acq_date
- confidence

These fields are sufficient to define where and when fire detections occurred.

In [ ]:
df_clean = df[['latitude', 'longitude', 'acq_date', 'confidence']].copy()
df_clean.head()

,latitude,longitude,acq_date,confidence
0,-34.46007,150.88153,2016-01-01,n
1,-34.45628,150.87708,2016-01-01,n
2,-34.46427,150.88800,2016-01-01,l
3,-33.79195,150.88683,2016-01-01,l
4,-34.46052,150.88356,2016-01-01,n


## Filter Low-Confidence Detections
NASA FIRMS confidence values are encoded as:
- l = low
- n = nominal
- h = high

Low-confidence detections are removed to reduce noise in the dataset.

In [ ]:
df_clean = df_clean[df_clean['confidence'].isin(['n', 'h'])].copy()
df_clean['confidence'].unique()

array(['n', 'h'], dtype=object)

## Create Positive Fire Label
All remaining rows in the FIRMS detection dataset represent fire detections, so a target label is added:
- fire_occurred = 1

In [ ]:
df_clean['fire_occurred'] = 1
df_clean.head()

,latitude,longitude,acq_date,confidence,fire_occurred
0,-34.46007,150.88153,2016-01-01,n,1
1,-34.45628,150.87708,2016-01-01,n,1
4,-34.46052,150.88356,2016-01-01,n,1
6,-31.36977,151.52815,2016-01-01,n,1
7,-25.76299,151.73648,2016-01-01,n,1


## Generate Negative Samples
To support binary classification, synthetic non-fire samples are generated.
These samples use random locations within Australia and sampled dates from the source dataset.

**Note:** These are proxy non-fire examples for MVP purposes and may be refined in later stages.

In [ ]:
num_samples = len(df_clean)

latitudes = np.random.uniform(-44, -10, num_samples)
longitudes = np.random.uniform(112, 154, num_samples)
dates = np.random.choice(df_clean['acq_date'], num_samples)

df_negative = pd.DataFrame({
    'latitude': latitudes,
    'longitude': longitudes,
    'acq_date': dates,
    'confidence': 'none',
    'fire_occurred': 0
})

df_negative.head()

,latitude,longitude,acq_date,confidence,fire_occurred
0,-27.552401,119.335439,2016-01-08,none,0
1,-17.129261,125.462675,2016-01-04,none,0
2,-28.873274,125.488677,2016-01-05,none,0
3,-13.224712,125.748227,2016-01-06,none,0
4,-30.261050,131.463439,2016-01-06,none,0


## Combine Positive and Negative Samples
The cleaned fire detections and generated non-fire samples are combined into one dataset and shuffled.

In [ ]:
df_final = pd.concat([df_clean, df_negative], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

df_final.head()

,latitude,longitude,acq_date,confidence,fire_occurred
0,-32.845390,116.004110,2016-01-06,h,1
1,-30.783996,144.905911,2016-01-05,none,0
2,-32.861230,116.076350,2016-01-06,n,1
3,-21.799580,117.842850,2016-01-08,n,1
4,-28.702169,123.860462,2016-01-03,none,0


## Validate Final Dataset
This step confirms:
- class balance
- dataset shape
- final structure

In [ ]:
print(df_final.columns)
print(df_final.shape)
print(df_final['fire_occurred'].value_counts())
df_final.sample(10)

Index(['latitude', 'longitude', 'acq_date', 'confidence', 'fire_occurred'], dtype='object')
(17670, 5)
fire_occurred
1    8835
0    8835
Name: count, dtype: int64


,latitude,longitude,acq_date,confidence,fire_occurred
14307,-32.870520,115.986240,2016-01-06,h,1
16886,-36.446727,131.210723,2016-01-06,none,0
3998,-17.021052,130.538693,2016-01-06,none,0
6827,-19.083870,122.683020,2016-01-08,n,1
12082,-21.738090,122.258050,2016-01-07,n,1
12820,-40.310318,125.891502,2016-01-06,none,0
17398,-22.403593,125.710404,2016-01-07,none,0
7181,-10.887888,118.061000,2016-01-01,none,0
12059,-33.065634,146.445573,2016-01-02,none,0
4739,-25.365881,147.504242,2016-01-03,none,0


## Save Final Dataset
The final cleaned dataset is exported as a CSV for downstream modelling.

In [ ]:
output_path = "fire_prediction_dataset.csv"
df_final.to_csv(output_path, index=False)
print(f"Saved dataset to: {output_path}")

Saved dataset to: fire_prediction_dataset.csv


## Conclusion
This notebook prepared a structured fire prediction dataset from NASA FIRMS data by:
- selecting relevant fire detection fields
- filtering low-confidence detections
- assigning positive fire labels
- generating synthetic non-fire samples
- combining and exporting a balanced dataset

This dataset can now be used in later stages for feature integration with weather and soil moisture data, and for bushfire prediction modelling.